In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("test-spark")
    .getOrCreate()
)

26/06/17 09:14:42 WARN Utils: Your hostname, fatima-HP-ZBook-15-G3 resolves to a loopback address: 127.0.1.1; using 10.26.1.46 instead (on interface wlp2s0)
26/06/17 09:14:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/17 09:14:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
df = spark.read.csv("../data/raw/olist_customers_dataset.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("../data/bronze/olist_customers_dataset.parquet")

df = spark.read.csv("../data/raw/olist_geolocation_dataset.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("../data/bronze/olist_geolocation_dataset.parquet")

df = spark.read.csv("../data/raw/olist_order_items_dataset.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("../data/bronze/olist_order_items_dataset.parquet")

df = spark.read.csv("../data/raw/olist_order_payments_dataset.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("../data/bronze/olist_order_payments_dataset.parquet")

df = spark.read.csv("../data/raw/olist_order_reviews_dataset.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("../data/bronze/olist_order_reviews_dataset.parquet")

df = spark.read.csv("../data/raw/olist_orders_dataset.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("../data/bronze/olist_orders_dataset.parquet")

df = spark.read.csv("../data/raw/olist_products_dataset.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("../data/bronze/olist_products_dataset.parquet")

df = spark.read.csv("../data/raw/olist_sellers_dataset.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("../data/bronze/olist_sellers_dataset.parquet")

df = spark.read.csv("../data/raw/product_category_name_translation.csv", header=True, inferSchema=True)
df.write.mode("overwrite").parquet("../data/bronze/product_category_name_translation.parquet")

In [8]:
#Convertir les fichiers en parquet

filename = {
    "olist_customers_dataset",
    "olist_geolocation_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_orders_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation"
}

for f in filename : 
    df = spark.read.csv(f"../data/raw/{f}.csv", header=True, inferSchema=True)
    df.write.mode("overwrite").parquet(f"../data/bronze/{f}.parquet")

26/06/17 09:39:30 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


In [ ]:
import os

def load_raw():
    
    spark = (
        SparkSession.builder
        .appName("load_raw")
        .getOrCreate()
    )

    raw_path = "../data/raw/"

    files = [f for f in os.listdir(raw_path) if f.endswith('.csv')]
    
    for file in files:

        df = spark.read.csv(os.path.join(raw_path, file), header=True, inferSchema=True)
        print(f"Nombre de lignes : {df.count()}")
        df.printSchema()

    spark.stop()

In [62]:
load_raw()

26/06/17 13:07:14 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Nombre de lignes : 104162
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)

Nombre de lignes : 32951
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)

Nombre de lignes : 103886
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: str

Nombre de lignes : 1000163
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

Nombre de lignes : 112650
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

Nombre de lignes : 71
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)

Nombre de lignes : 99441
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- custome

In [70]:
import os

def to_bronze():

    spark = (
        SparkSession.builder
        .appName("to_bronze")
        .getOrCreate()
    )

    raw_path = "../data/raw/"
    bronze_path = "../data/bronze"

    if not os.path.exists(bronze_path):
        os.makedirs(bronze_path)

    files = [f for f in os.listdir(raw_path) if f.endswith('.csv')]
    
    for file in files:

        df = spark.read.csv(os.path.join(raw_path, file), header=True, inferSchema=True)
        output_file = os.path.join(bronze_path, f"{file[:-4]}.parquet")
        df.write.mode("overwrite").parquet(output_file)
        print(f"Fichier enregistré {output_file}")

    spark.stop()

In [71]:
to_bronze()

Fichier enregistré ../data/bronze/olist_order_reviews_dataset.parquet
Fichier enregistré ../data/bronze/olist_products_dataset.parquet
Fichier enregistré ../data/bronze/olist_order_payments_dataset.parquet
Fichier enregistré ../data/bronze/olist_sellers_dataset.parquet


26/06/17 13:21:14 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


Fichier enregistré ../data/bronze/olist_geolocation_dataset.parquet
Fichier enregistré ../data/bronze/olist_order_items_dataset.parquet
Fichier enregistré ../data/bronze/product_category_name_translation.parquet
Fichier enregistré ../data/bronze/olist_customers_dataset.parquet


Fichier enregistré ../data/bronze/olist_orders_dataset.parquet


In [15]:
# Lire les fichiers parquet 

df_customers = spark.read.format('parquet') \
        .option('inferSchema', True) \
        .option('header', True) \
        .load('../data/bronze/olist_customers_dataset.parquet')

df_geolocation = spark.read.format('parquet') \
        .option('inferSchema', True) \
        .option('header', True) \
        .load('../data/bronze/olist_geolocation_dataset.parquet')

df_items = spark.read.format('parquet') \
        .option('inferSchema', True) \
        .option('header', True) \
        .load('../data/bronze/olist_order_items_dataset.parquet')


df_payments = spark.read.format('parquet') \
        .option('inferSchema', True) \
        .option('header', True) \
        .load('../data/bronze/olist_order_payments_dataset.parquet')

df_reviews = spark.read.format('parquet') \
        .option('inferSchema', True) \
        .option('header', True) \
        .load('../data/bronze/olist_order_reviews_dataset.parquet')

df_orders = spark.read.format('parquet') \
        .option('inferSchema', True) \
        .option('header', True) \
        .load('../data/bronze/olist_orders_dataset.parquet')


df_products = spark.read.format('parquet') \
        .option('inferSchema', True) \
        .option('header', True) \
        .load('../data/bronze/olist_products_dataset.parquet')


df_sellers = spark.read.format('parquet') \
        .option('inferSchema', True) \
        .option('header', True) \
        .load('../data/bronze/olist_sellers_dataset.parquet')


df_category = spark.read.format('parquet') \
        .option('inferSchema', True) \
        .option('header', True) \
        .load('../data/bronze/product_category_name_translation.parquet')

In [31]:
df_category.show()

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza_saude|                health_beauty|
| informatica_acess...|         computers_accesso...|
|           automotivo|                         auto|
|      cama_mesa_banho|               bed_bath_table|
|     moveis_decoracao|              furniture_decor|
|        esporte_lazer|               sports_leisure|
|           perfumaria|                    perfumery|
| utilidades_domest...|                   housewares|
|            telefonia|                    telephony|
|   relogios_presentes|                watches_gifts|
|    alimentos_bebidas|                   food_drink|
|                bebes|                         baby|
|            papelaria|                   stationery|
| tablets_impressao...|         tablets_printing_...|
|           brinquedos|                         toys|
|       telefonia_fixa|     

In [39]:
# Afficher le nombre de lignes de chaque dataframe

all_df = {
    "df_customers": df_customers,
    "df_geolocation": df_geolocation,
    "df_items": df_items,
    "df_payments": df_payments,
    "df_reviews": df_reviews,
    "df_orders": df_orders,
    "df_products": df_products,
    "df_sellers": df_sellers,
    "df_category": df_category,
}

for df_name, df in all_df.items():
    print(f"Nombre de lignes pour {df_name} :", df.count())


Nombre de lignes pour df_customers : 99441
Nombre de lignes pour df_geolocation : 1000163
Nombre de lignes pour df_items : 112650
Nombre de lignes pour df_payments : 103886
Nombre de lignes pour df_reviews : 104162
Nombre de lignes pour df_orders : 99441
Nombre de lignes pour df_products : 32951
Nombre de lignes pour df_sellers : 3095
Nombre de lignes pour df_category : 71


In [40]:
# Afficher les colonnes 

all_df = {
    "df_customers": df_customers,
    "df_geolocation": df_geolocation,
    "df_items": df_items,
    "df_payments": df_payments,
    "df_reviews": df_reviews,
    "df_orders": df_orders,
    "df_products": df_products,
    "df_sellers": df_sellers,
    "df_category": df_category,
}

for df_name, df in all_df.items():
    print(f"Colonnes {df_name} :", df.columns)


Colonnes df_customers : ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
Colonnes df_geolocation : ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']
Colonnes df_items : ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
Colonnes df_payments : ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
Colonnes df_reviews : ['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']
Colonnes df_orders : ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Colonnes df_products : ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'pro

In [43]:
all_df = [
    df_customers,
    df_geolocation,
    df_items,
    df_payments,
    df_reviews,
    df_orders,
    df_products,
    df_sellers,
    df_category,
]

for df in all_df :
    df.printSchema()


root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable 

In [55]:
print("Toutes les lignes ", df_customers.count())
print("Les lignes uniques ", df_customers.select("customer_id").distinct().count())

Toutes les lignes  99441
Les lignes uniques  99441


In [56]:
print("Toutes les lignes ", df_customers.count())
print("Les lignes uniques ", df_customers.select("customer_unique_id").distinct().count())

Toutes les lignes  99441
Les lignes uniques  96096


In [57]:
print("Toutes les lignes ", df_geolocation.count())
print("Les lignes uniques ", df_geolocation.select("geolocation_zip_code_prefix").distinct().count())

Toutes les lignes  1000163
Les lignes uniques  19015


In [59]:
df_geolocation_csv = spark.read.format('csv') \
        .option('inferSchema', True) \
        .option('header', True) \
        .load('../data/raw/olist_geolocation_dataset.csv')

In [60]:
print("Toutes les lignes ", df_geolocation_csv.count())
print("Les lignes uniques ", df_geolocation_csv.select("geolocation_zip_code_prefix").distinct().count())

Toutes les lignes  1000163


Les lignes uniques  19015


26/06/17 12:58:09 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 3395140 ms exceeds timeout 120000 ms
26/06/17 12:58:10 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/17 12:58:10 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

In [ ]:
print("Toutes les lignes ", df_items.count())
print("Les lignes uniques ", df_items.select("order_item_id").distinct().count())

Toutes les lignes  112650
Les lignes uniques  21


In [78]:
def to_sliver():

    spark = SparkSession.builder \
        .appName("OlistSilverCleaning") \
        .getOrCreate()

    bronze_path = "../data/bronze"
    silver_path = "../data/silver"

    if not os.path.exists(silver_path):
        os.makedirs(silver_path)

    files = [f for f in os.listdir(bronze_path) if f.endswith('.parquet')]

    for file in files:
        df = spark.read.parquet(os.path.join(bronze_path, file), header=True, inferSchema=True)
        df = df.dropDuplicates()
        output_file = os.path.join(silver_path, f"{file}")
        df.write.mode("overwrite").parquet(output_file)
        print(f"Fichier enregistré {output_file}")

    spark.stop()

In [79]:
to_sliver()

Fichier enregistré ../data/silver/olist_order_payments_dataset.parquet
Fichier enregistré ../data/silver/olist_customers_dataset.parquet
Fichier enregistré ../data/silver/olist_sellers_dataset.parquet
Fichier enregistré ../data/silver/product_category_name_translation.parquet


26/06/17 13:53:37 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


Fichier enregistré ../data/silver/olist_order_items_dataset.parquet
Fichier enregistré ../data/silver/olist_products_dataset.parquet


26/06/17 13:53:38 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


Fichier enregistré ../data/silver/olist_order_reviews_dataset.parquet


26/06/17 13:53:40 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


Fichier enregistré ../data/silver/olist_orders_dataset.parquet


26/06/17 13:53:42 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


Fichier enregistré ../data/silver/olist_geolocation_dataset.parquet


In [91]:
spark = SparkSession.builder \
        .appName("geolocation") \
        .getOrCreate()

df_geolocation_silver = spark.read.parquet('../data/silver/olist_geolocation_dataset.parquet')
df_geolocation_bronze = spark.read.parquet('../data/bronze/olist_geolocation_dataset.parquet')

print(df_geolocation_silver.count())
print(df_geolocation_bronze.count())

738332
1000163


In [105]:
from pyspark.sql.functions import col, column 

df_order_item = spark.read.parquet('../data/silver/olist_order_items_dataset.parquet')
df_order_item.select(column('price')).show()

+------+
| price|
+------+
|  40.0|
|  59.9|
| 63.72|
| 76.99|
| 34.84|
| 119.9|
| 170.0|
| 17.99|
|  99.0|
| 325.0|
| 115.0|
| 15.75|
|  98.0|
|110.32|
| 56.99|
|174.92|
|  34.0|
|  59.9|
|166.99|
|  75.0|
+------+
only showing top 20 rows



In [119]:
df_order_item = spark.read.parquet('../data/silver/olist_order_reviews_dataset.parquet')
df_order_item.select('review_answer_timestamp').show()
df_order_item.columns


+-----------------------+
|review_answer_timestamp|
+-----------------------+
|    2018-08-15 06:07:15|
|    2018-05-17 12:14:05|
|    2018-08-24 18:29:24|
|    2018-08-14 14:25:20|
|    2018-06-27 13:13:29|
|    2018-05-02 20:25:19|
|    2018-08-28 14:25:19|
|    2018-05-28 18:12:17|
|    2018-07-31 01:18:56|
|    2018-07-10 11:13:40|
|    2018-04-26 18:07:59|
|    2018-08-23 15:18:55|
|    2018-06-21 13:15:20|
|    2018-05-13 03:38:46|
|    2018-08-02 10:17:31|
|    2018-07-25 20:20:01|
|    2018-07-01 02:32:34|
|    2018-05-08 11:24:16|
|    2018-04-28 17:16:26|
|    2018-06-22 12:28:09|
+-----------------------+
only showing top 20 rows



['review_id',
 'order_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp']

In [ ]:
df = spark.read.parquet('../data/silver/olist_sellers_dataset.parquet.parquet')
#df_order_item = df_order_item.withColumn("timestamp_col", col("string_col").cast("timestamp"))
df.column

In [ ]:
from pyspark.sql.functions import col, trim, lower

bronze_path = "../data/bronze"

df_sell = spark.read.parquet(os.path.join(bronze_path, "olist_sellers_dataset.parquet"))
    
df_sell = df_sell.withColumn("seller_city", trim(lower(col("seller_city")))) \
                 .withColumn("seller_state", trim(col("seller_state"))) 

df_cust = spark.read.parquet(os.path.join(bronze_path, "olist_customers_dataset.parquet"))

+--------------------+----------------------+-----------------+------------+
|           seller_id|seller_zip_code_prefix|      seller_city|seller_state|
+--------------------+----------------------+-----------------+------------+
|3442f8959a84dea7e...|                 13023|         campinas|          SP|
|d1b65fc7debc3361e...|                 13844|       mogi guacu|          SP|
|ce3ad9de960102d06...|                 20031|   rio de janeiro|          RJ|
|c0f3eea2e14555b6f...|                  4195|        sao paulo|          SP|
|51a04a8a6bdcb23de...|                 12914|braganca paulista|          SP|
|c240c4061717ac180...|                 20920|   rio de janeiro|          RJ|
|e49c26c3edfa46d22...|                 55325|           brejao|          PE|
|1b938a7ec6ac5061a...|                 16304|        penapolis|          SP|
|768a86e36ad6aae3d...|                  1529|        sao paulo|          SP|
|ccc4bbb5f32a6ab2b...|                 80310|         curitiba|          PR|

In [ ]:
df_cust = df_cust.withColumn("seller_city", trim(lower(col("seller_city")))) \
                 .withColumn("seller_state", trim(col("seller_state"))) 